In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# just making sure these are available
print('imports done')

imports done


---
## Q1 – Dataset Splitting

The question says: 10 genres, 100 recipe dicts each. Pass the combined list into `train_test_split` with `test_size=0.2, shuffle=True, random_state=42`. How many items go into the val set?

This is pretty straightforward. Total = 10 * 100 = 1000. 20% of 1000 = 200. Let me just confirm with code.

In [15]:
# simulating the recipe list — each recipe is just a dummy dict here
# structure doesn't matter, only the count does
genres = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']

all_recipes = []
for genre in genres:
    for i in range(100):
        all_recipes.append({'genre': genre, 'song_id': i})

print(f'total recipes: {len(all_recipes)}')

train_recipes, val_recipes = train_test_split(
    all_recipes,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print(f'train: {len(train_recipes)}')
print(f'val:   {len(val_recipes)}')

total recipes: 1000
train: 800
val:   200


---
## Q2 – On-the-Fly Mixing Shape

So we load 4 stems + 1 noise file using `librosa.load(sr=16000, duration=10)`. 16000 samples/sec × 10 sec = 160,000 samples. All arrays get padded/truncated to exactly 160,000.

Then: mix = sum of 4 stems + noise * 0.2

Librosa returns 1D mono arrays by default. So summing 1D arrays gives you another 1D array. The shape should just be (160000,).

Let me verify this logic:

In [16]:
TARGET_LEN = 16000 * 10  # sr * duration
print(f'target length: {TARGET_LEN}')

# simulating what librosa.load gives us (1D arrays)
def fake_load(length=TARGET_LEN):
    return np.random.randn(length).astype(np.float32)

def pad_or_truncate(arr, length):
    if len(arr) >= length:
        return arr[:length]
    else:
        return np.pad(arr, (0, length - len(arr)))

stems = [pad_or_truncate(fake_load(), TARGET_LEN) for _ in range(4)]
noise = pad_or_truncate(fake_load(), TARGET_LEN)

# mixing step
mix = stems[0] + stems[1] + stems[2] + stems[3] + noise * 0.2

print(f'mix shape: {mix.shape}')
print(f'mix dtype: {mix.dtype}')

target length: 160000
mix shape: (160000,)
mix dtype: float32


---
## Q3 – Hugging Face Feature Extractor Shape

This one requires actually running the AST feature extractor. The model is `MIT/ast-finetuned-audioset-10-10-0.4593`.

The steps:
1. Initialize `AutoFeatureExtractor` from that checkpoint
2. Create `mix = np.ones(160000)`
3. Pass to extractor with `sampling_rate=16000`, `return_tensors="pt"`
4. Get `input_values`, apply `.squeeze(0)` to remove batch dim
5. Check shape

AST converts audio to log-mel spectrogram. For 10 seconds at sr=16000, i know the output is a 2D spectrogram — time frames × mel bins. Let me run it.

In [17]:
# installing transformers if not already there
!pip install transformers -q

In [18]:
from transformers import AutoFeatureExtractor
import torch

CHECKPOINT = 'MIT/ast-finetuned-audioset-10-10-0.4593'

feature_extractor = AutoFeatureExtractor.from_pretrained(CHECKPOINT)
print('feature extractor loaded')
print(f'sampling rate expected: {feature_extractor.sampling_rate}')

feature extractor loaded
sampling rate expected: 16000


In [19]:
# dummy mix as described in the question
mix = np.ones(160000)

inputs = feature_extractor(
    mix,
    sampling_rate=16000,
    return_tensors='pt'
)

input_values = inputs['input_values'].squeeze(0)

print(f'input_values shape after squeeze: {input_values.shape}')
# this is the spectrogram that AST will process

input_values shape after squeeze: torch.Size([1024, 128])


---
## Q4 – Trainable Parameters in the AST Model

Now we load `ASTForAudioClassification` with `num_labels=10` and `ignore_mismatched_sizes=True` (since the original model was trained on 527 AudioSet classes, not 10).

Then count trainable params with `sum(p.numel() for p in model.parameters() if p.requires_grad)`

In [20]:
from transformers import ASTForAudioClassification

model = ASTForAudioClassification.from_pretrained(
    CHECKPOINT,
    num_labels=10,
    ignore_mismatched_sizes=True
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'trainable parameters: {trainable_params}')

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable parameters: 86196490


---
## Q5 – Inference Normalization

Formula: `y = y / (np.max(np.abs(y)) + 1e-9)`

Test input: `y_test = np.array([-0.85, 0.40, 0.20, -0.10])`

Max absolute value = 0.85 (from -0.85). So y[0] = -0.85 / (0.85 + 1e-9) ≈ -1.000

Pretty straightforward, let me just confirm:

In [21]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])

print(f'max abs value: {np.max(np.abs(y_test))}')

# apply the formula
y_norm = y_test / (np.max(np.abs(y_test)) + 1e-9)

print(f'normalized array: {y_norm}')
print(f'index 0 value (rounded to 3dp): {round(y_norm[0], 3)}')

max abs value: 0.85
normalized array: [-1.          0.47058823  0.23529412 -0.11764706]
index 0 value (rounded to 3dp): -1.0


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.
